# Combining MC + data

Covers adding on-beam & off-beam data on top of step1's MC-only setup,
plus POT/gate accounting. For adding a MeVPrtl/HNL sample on top of
this, see `step2b_combining_mc_hnl_and_data.ipynb`.

| Section | File | What you set |
|---|---|---|
| A1 | -- | produce `DATA_FILE`/`OFFBEAM_FILE` via `run_df_maker.py` |
| A2 | `analysis.py` | `offbeam` category |
| A3 | `preprocess.py` | on-beam/off-beam fixes (optional) |
| A4 | -- | load MC + on-beam + off-beam (cuts applied) + off-beam scaling |
| A5 | `plotting.py` | plot MC + data with `plot_mc_data` |

In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys
# TODO: set to the PARENT directory of your cafpybara/ clone -- NOT cafpybara/ itself (skipped if already importable).
sys.path.append("TODO_SET_ME/parent_dir_of_cafpybara")

# TODO: set to your cafpyana clone (skipped if already exported).
os.environ.setdefault("CAFPYANA_PATH", "TODO_SET_ME/path/to/cafpyana")

import numpy as np
import pandas as pd
from cafpybara import core

# TODO: import your own analysis's name instead of _template.
import cafpybara.analyses._template as ana

STILL_ON_TEMPLATE = ana.__name__.endswith('_template')
if STILL_ON_TEMPLATE:
    print("Still importing _template directly -- see the TODO above.")
else:
    print("Imported cleanly -- a real, importable analysis.")

## A1. Make `DATA_FILE`/`OFFBEAM_FILE` with `cafpyana`

Same tool as step1's `MC_FILE` (no-preselection config), run against
on-beam and off-beam input lists:

1. `cd $CAFPYANA_PATH`
2. ```bash
   python run_df_maker.py -c ./analysis_village/hnl_nuee_nupi0/configs/hnl_data_nopreselect_savepfp.py -l <input.list> -o <output_name>
   ```

NOTE: run `concat_hdf.py` after producing per-shard dataframes -- the
resulting `file_idx` column is required to prevent duplication when
computing `mcstat` (section A4).

**TODO:** set `MC_FILE`, `DATA_FILE`, `OFFBEAM_FILE`

In [ ]:
#MC_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/mcbnb_noselection.df"
MC_FILE = "/path/to/your_mc_file.h5"
#DATA_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/dtbnb_noselection.df"
DATA_FILE = "/path/to/your_data_file.h5"
#OFFBEAM_FILE = "/exp/sbnd/data/users/lnguyen/cafpyana_pi0/dataframes/demo/dtoff_noselection.df"
OFFBEAM_FILE = "/path/to/your_offbeam_file.h5"
#VAR = "your_variable"
#BINS = [0, 1, 2, 3]
VAR = ('pfp', 'shw', 'bestplane_energy', '', '', '') #TEST
BINS = np.linspace(0, 1, 10) #TEST

## A2. Set offbeam category in `analysis.py`

* **TODO:** add an `"offbeam"` entry to `signal_categories` in `/my_analysis/analysis.py`

Example: add the category (`analysis.py`)

```python
signal_categories = {
    ...,
    "offbeam": {"value": <next_int>, "label": "Offbeam", "color": "#D0D2D4"},
}
signal_dict = {k: v["value"] for k, v in signal_categories.items()}
```

## A3. Add on-beam/off-beam fixes in `preprocess.py` (Optional)

* **TODO (optional):**
  * Add new on-beam/off-beam-specific functions (not edits to the generic
    `preprocess_data`) in `/my_analysis/preprocess.py`. Pass them
    explicitly as `preprocess_fn=` to `load_data` in section A4 -- an
    unset one there falls back to the no-op `preprocess_data`.
  * add both new function names to `preprocess.py`'s `__all__` -- unlike
    editing `preprocess_mc` in place (step1a's A6), these are brand-new
    names; skip this and `ana.preprocess_databnb`/`ana.preprocess_dataoff`
    raise `AttributeError` even though the functions exist in the module.

Example: On-beam + off-beam timing calibration (`analyses/hnlpi0/preprocess.py`)

```python
from ...core.preprocess import fix_databnb_timing_calibration, fix_timing_calibration
from ...core import timing_calibration as tc

__all__ = ['preprocess_mc', 'preprocess_data', 'preprocess_databnb', 'preprocess_dataoff']

def preprocess_databnb(df):
    df = preprocess_data(df)
    df = fix_databnb_timing_calibration(df)  # drops bad-period rows,
                                              # corrects each good period
    return df

def preprocess_dataoff(df):
    df = preprocess_data(df)
    df = fix_timing_calibration(df, period=tc.offbeam_period_calib,
                                 t0_offset=tc.offbeam_offset_calib,
                                 ifData=True)
    return df
```

## A4. Load MC + data with cuts applied, then off-beam scaling

Single load step -- section A5 just plots whatever comes out of this.

* **TODO:**
  * pass `preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data)`
    to the on-beam call, and the off-beam equivalent to the off-beam call --
    only takes effect if you added section A3's functions, otherwise both
    fall back to the no-op `preprocess_data`.
  * double-check `f` is the most up-to-date off-beam scaling fudge factor
    (see SBN-docdb-41013 & SBN-docdb-43255).
  * weight: `1.0` on `mc_df`, `scale_offbeam` on `offbeam_df`.
  * concat: `pd.concat` them so `plot_mc_data` stacks off-beam alongside
    the other MC backgrounds -- on-beam `data_df` stays the separate overlay.
* `ana.mcstat` adds bootstrapped MC statistical-uncertainty universes to
  `mc_df` before the merge.

Example: HNL/pi0's merge (`analyses/hnlpi0/examples/hnl_plotting.ipynb`)

In [ ]:
try:
    mc_df, mc_pot, mc_ngen = ana.load_mc(MC_FILE, cuts=ana.DEFAULT_CUTS)

    data_df, data_pot, data_ngates = ana.load_data(
        DATA_FILE, cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_databnb', ana.preprocess_data),
    )

    offbeam_df, offbeam_pot, offbeam_ngates = ana.load_data(
        OFFBEAM_FILE, onbeam=False,
        offbeam_signal_value=ana.signal_dict["offbeam"],  # TODO above -- add this category first (A2)
        cuts=ana.DEFAULT_CUTS,
        preprocess_fn=getattr(ana, 'preprocess_dataoff', ana.preprocess_data),
    )

    print("MC POT:        ", mc_pot)
    print("Data POT:      ", data_pot)
    print("Data gates:    ", data_ngates)
    print("Offbeam gates: ", offbeam_ngates)

    f = 0.0725  # TODO: double check for the most-up-to-date fudge factor
    ongates_per_pot = data_ngates / data_pot
    scale_offbeam = ((1 - f) * (ongates_per_pot * mc_pot)) / offbeam_ngates
    print("Offbeam scale: ", scale_offbeam)

    weight_col = ('weights_mc', '', '', '', '', '')
    mc_df[weight_col] = 1.0
    offbeam_df[weight_col] = scale_offbeam

    mcstat_cols = ['__ntuple', 'entry', 'rec.slc..index', 'run', 'subrun', 'evt', 'sample']
    mc_top_level = set(mc_df.columns.get_level_values(0)) | set(mc_df.index.names)
    if 'file_idx' in mc_top_level:
        mcstat_cols.append('file_idx')  # only from concat_hdf.py (step5) -- fine if absent here

    mc_df = pd.concat([
        ana.mcstat(mc_df.assign(sample=0).set_index("sample", append=True).reset_index(), cols=mcstat_cols),
        offbeam_df.assign(sample=1).set_index("sample", append=True).reset_index(),
    ])
    print("Merged MC + off-beam rows:", len(mc_df))
except Exception as e:
    mc_df, data_df = None, None
    print("No real MC/data/offbeam file here (this notebook needs none) -- expected:")
    print(f"  {type(e).__name__}: {e}")

## A5. Plot MC + data with `ana.plot_mc_data`

* Composes `plot_var` (MC stack) with a data overlay and a ratio panel.
* **TODO (optional):**
  * inspect `plot_mc_data`/`plot_var`'s docstring for plotting options
    (labels, systematics, legend, etc.)
  * for PDG/mode breakdown plots: `pdg_categories`/`mode_categories`
    default to `core.physics`'s generic ones; `pdg_col` is passed per
    call, not a module default

In [ ]:
if mc_df is not None and data_df is not None:
    #mc_df (merged with off-beam)/data_df loaded with cuts at section A4 above
    ana.plot_mc_data(mc_df, data_df, VAR, bins=BINS, scale=data_pot / mc_pot)
else:
    print("No real data file here (this notebook needs none) -- see section A4 above")